# Data Cleaning Pipeline

**Mục tiêu:** Làm sạch dữ liệu tín dụng trước khi xây dựng mô hình phân loại nhóm nợ (NHOMNOMOI).

**Các bước xử lý:**

| Bước | Cột xử lý | Mô tả |
|------|-----------|-------|
| 1 | CURRENCYCD | Loại bỏ hàng USD, xóa cột |
| 2 | DESC_TIME, THOIHAN | Lọc theo ngưỡng hợp lệ, chia 30 với vay ngắn hạn |
| 3 | LAISUAT | Loại bỏ lãi suất > 40 |
| 4 | MUCDICHVAY | Giữ 4 ký tự đầu |
| 5 | PARENTORGNAME | Trích tên chi nhánh sau chữ 'CN ' |
| 6 | BASE_BAL, DUNO_QD | Winsorize về khoảng P20 - P80 |

---
## 1. Import thư viện

In [ ]:
#pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', None)

---
## 2. Load dữ liệu

In [ ]:
df = pd.read_csv('data_SEX_Solving.csv')

print('Số dòng:', len(df))
print('Số cột :', len(df.columns))

In [ ]:
# Phân phối biến mục tiêu
print('Phân phối NHOMNOMOI:')
print(df['NHOMNOMOI'].value_counts().sort_index())

In [ ]:
# Xem mẫu dữ liệu
df.head()

---
## 3. Hàm xử lý dữ liệu (process_data)

In [ ]:
def process_data(df):
    df = df.copy()

    # Bước 1: Xử lý CURRENCYCD
    # Loại bỏ các hàng có đồng tiền USD, sau đó xóa cột
    if 'CURRENCYCD' in df.columns:
        df = df[df['CURRENCYCD'] != 'USD']
        df.drop(columns=['CURRENCYCD'], inplace=True)

    # Bước 2: Xử lý DESC_TIME và THOIHAN
    # Ép kiểu số để tính toán
    df['THOIHAN'] = pd.to_numeric(df['THOIHAN'], errors='coerce')

    # Chỉ giữ lại các dòng nằm trong khoảng hợp lệ của từng loại vay
    mask_dai_han   = (df['DESC_TIME'] == 'Vay dai han')   & (df['THOIHAN'] >  60) & (df['THOIHAN'] <= 400)
    mask_trung_han = (df['DESC_TIME'] == 'Vay trung han') & (df['THOIHAN'] >= 13) & (df['THOIHAN'] <= 61)
    mask_ngan_han  = (df['DESC_TIME'] == 'Vay ngan han')  & (
        ((df['THOIHAN'] >= 0)  & (df['THOIHAN'] <= 13)) |
        ((df['THOIHAN'] >= 30) & (df['THOIHAN'] <= 366))
    )

    df = df[mask_dai_han | mask_trung_han | mask_ngan_han]

    # Chuyển THOIHAN vay ngắn hạn từ ngày sang tháng (chia 30)
    mask_to_divide = (
        (df['DESC_TIME'] == 'Vay ngan han') &
        (df['THOIHAN'] >= 30) &
        (df['THOIHAN'] <= 366)
    )
    df.loc[mask_to_divide, 'THOIHAN'] = (df.loc[mask_to_divide, 'THOIHAN'] / 30).astype(int)

    # Bước 3: Xử lý LAISUAT
    # Loại bỏ các mức lãi suất bất thường (> 40%)
    df['LAISUAT'] = pd.to_numeric(df['LAISUAT'], errors='coerce')
    df = df[df['LAISUAT'] <= 40]

    # Bước 4: Xử lý MUCDICHVAY
    # Chỉ lấy 4 ký tự đầu để chuẩn hóa mã mục đích
    df['MUCDICHVAY'] = df['MUCDICHVAY'].astype(str).str[:4]

    # Bước 5: Xử lý PARENTORGNAME
    # Trích tên chi nhánh: 'KLB - CN KHANH HOA' -> 'KHANH HOA'
    def extract_branch(name):
        if isinstance(name, str) and 'CN ' in name:
            return name.split('CN ', 1)[1]
        return name

    df['PARENTORGNAME'] = df['PARENTORGNAME'].apply(extract_branch)

    # Bước 6: Winsorize BASE_BAL và DUNO_QD (P20 - P80)
    # Giới hạn giá trị về khoảng percentile 20-80 để hạn chế outlier
    for col in ['BASE_BAL', 'DUNO_QD']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            lower = df[col].quantile(0.20)
            upper = df[col].quantile(0.80)
            df[col] = df[col].clip(lower=lower, upper=upper)
            print(f'[{col}] Winsorize: lower = {lower:>15,.0f} | upper = {upper:>15,.0f}')

    return df


df_clean = process_data(df)

In [ ]:
# Kiểm tra dữ liệu sau xử lý
df_clean.head()

---
## 4. Thống kê kết quả làm sạch theo nhóm nợ

In [ ]:
data_list = []

for i in range(1, 6):
    so_mau_ban_dau = len(df[df['NHOMNOMOI'] == i])
    so_mau_luc_sau = len(df_clean[df_clean['NHOMNOMOI'] == i])
    so_luong_xoa   = so_mau_ban_dau - so_mau_luc_sau
    ty_le_xoa      = so_luong_xoa / so_mau_ban_dau

    data_list.append({
        'Nhóm'            : i,
        'Số Mẫu Ban Đầu'  : so_mau_ban_dau,
        'Số Mẫu Lúc Sau'  : so_mau_luc_sau,
        'Số Lượng Xóa'    : so_luong_xoa,
        'Tỷ Lệ Xóa (%)'   : round(ty_le_xoa * 100, 2)
    })

df_ketqua = pd.DataFrame(data_list)

# Tính dòng tổng
tong = {
    'Nhóm'           : 'TỔNG',
    'Số Mẫu Ban Đầu' : df_ketqua['Số Mẫu Ban Đầu'].sum(),
    'Số Mẫu Lúc Sau' : df_ketqua['Số Mẫu Lúc Sau'].sum(),
    'Số Lượng Xóa'   : df_ketqua['Số Lượng Xóa'].sum(),
    'Tỷ Lệ Xóa (%)' : round(
        df_ketqua['Số Lượng Xóa'].sum() / df_ketqua['Số Mẫu Ban Đầu'].sum() * 100, 2
    )
}

df_ketqua = pd.concat([df_ketqua, pd.DataFrame([tong])], ignore_index=True)

pd.set_option('display.float_format', '{:,.2f}'.format)
df_ketqua

---
## 5. Xuất file

In [ ]:
df_clean.to_csv('xuly1.csv', index=False)

print('Đã lưu file: xuly1.csv')
print('Số dòng    :', len(df_clean))
print('Số cột     :', len(df_clean.columns))
print('Các cột    :', df_clean.columns.tolist())